In [1]:
import pandas as pd
vitals_df = pd.read_csv("vitals_final.csv")

/tmp/ipykernel_1416/1345775909.py:2: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  vitals_df = pd.read_csv("vitals_final.csv")


In [36]:
ed_only = vitals_df[vitals_df['charttime_ed'].isna() == False]
ed_only = ed_only.drop(columns = ['charttime_lab', 'Lab Oxygen Saturation %', 'Lab Temperature (C)', 'Hosp BMI (kg/m2)', 'Hosp Height (Inches)', 'Hosp Weight (Lbs)', 'Hosp Blood Pressure'])

ecg_df = pd.read_csv('ecg_data.csv')
ecg_ed_only = ecg_df[ecg_df['in_ed'] == 1]

clinical_encounters = pd.read_csv('clinical_encounters.csv')
ed_only_encounters = clinical_encounters[clinical_encounters['ed_stay_id'].isna() == False]

ed_only_encounters = ed_only_encounters.drop(columns = ['icu_outtime'])
ecg_ed_only = ecg_ed_only.drop(columns = ['icu_within_12hrs', 'icu_within_24hrs'])
ed_only_encounters = ed_only_encounters.drop(columns = ['icu_stay_id', 'icu_stay_id', 'icu_intime', 'icu_count'])

join1 = ecg_ed_only.merge(ed_only_encounters, how = 'left', on = ['subject_id', 'ed_stay_id'])
join1 = ecg_ed_only.merge(ed_only_encounters, how = 'left', on = ['subject_id', 'ed_stay_id'])

ecg = ecg_ed_only.copy()
enc = ed_only_encounters.copy()
vitals = ed_only.copy()

ecg['ecg_time'] = pd.to_datetime(ecg['ecg_time'])
vitals['charttime_ed'] = pd.to_datetime(vitals['charttime_ed'])
enc['ed_intime'] = pd.to_datetime(enc['ed_intime'])
enc['ed_outtime'] = pd.to_datetime(enc['ed_outtime'])

ecg = ecg[
    (ecg['in_ed'] == 1) &
    (ecg['ed_stay_id'].notna()) &
    (ecg['ecg_time'].notna())
].copy()

ecg = ecg.merge(
    enc[['subject_id', 'ed_stay_id', 'anchor_age', 'gender', 'race', 'ed_intime', 'ed_outtime']],
    on=['subject_id', 'ed_stay_id'],
    how='left'
)

vitals = vitals.rename(columns={'stay_id': 'ed_stay_id'})
vitals = vitals[
    [
        'subject_id',
        'ed_stay_id',
        'charttime_ed',
        'ED Temperature (F)',
        'ED Heart Rate',
        'ED Respitory Rate',
        'ED Oxygen Saturation %',
        'ED sbp',
        'ED dbp'
    ]
].copy()

ecg = ecg.sort_values('ecg_time')
vitals = vitals.sort_values('charttime_ed')

ecg_vitals = pd.merge_asof(
    ecg,
    vitals,
    left_on='ecg_time',
    right_on='charttime_ed',
    by=['subject_id', 'ed_stay_id'],
    direction='backward',
    tolerance=pd.Timedelta('4H')  # optional but recommended
)



/tmp/ipykernel_1416/2687285161.py:7: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  clinical_encounters = pd.read_csv('clinical_encounters.csv')


In [37]:
ecg_vitals

,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,in_hosp,in_ed,...,race,ed_intime,ed_outtime,charttime_ed,ED Temperature (F),ED Heart Rate,ED Respitory Rate,ED Oxygen Saturation %,ED sbp,ED dbp
0,10488072,41786232,41786232,2110-01-13 04:46:00,files/p1048/p10488072/s41786232/41786232,NaN,32221725.0,NaN,0,1,...,WHITE,2110-01-12 23:39:00,2110-01-13 10:52:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,10101340,42195598,42195598,2110-01-18 09:41:00,files/p1010/p10101340/s42195598/42195598,22311569.0,35530181.0,NaN,0,1,...,WHITE,2110-01-18 09:28:00,2110-01-18 21:40:00,2110-01-18 09:29:00,98.2,98.0,20.0,94.0,170.0,116.0
2,10101340,48109767,48109767,2110-01-18 10:19:00,files/p1010/p10101340/s48109767/48109767,22311569.0,35530181.0,NaN,0,1,...,WHITE,2110-01-18 09:28:00,2110-01-18 21:40:00,2110-01-18 09:29:00,98.2,98.0,20.0,94.0,170.0,116.0
3,10101340,48220829,48220829,2110-01-18 14:47:00,files/p1010/p10101340/s48220829/48220829,22311569.0,35530181.0,NaN,0,1,...,WHITE,2110-01-18 09:28:00,2110-01-18 21:40:00,2110-01-18 14:20:00,NaN,89.0,16.0,96.0,147.0,90.0
4,13198882,47647582,47647582,2110-01-24 18:17:00,files/p1319/p13198882/s47647582/47647582,25824712.0,37918046.0,NaN,0,1,...,WHITE,2110-01-24 16:46:00,2110-01-25 10:31:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75222,10079117,41685428,41685428,2210-09-15 09:22:00,files/p1007/p10079117/s41685428/41685428,NaN,32275589.0,NaN,0,1,...,BLACK/AFRICAN AMERICAN,2210-09-15 09:14:00,2210-09-15 15:52:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN
75223,10079117,41579618,41579618,2210-09-15 14:41:00,files/p1007/p10079117/s41579618/41579618,NaN,32275589.0,NaN,0,1,...,BLACK/AFRICAN AMERICAN,2210-09-15 09:14:00,2210-09-15 15:52:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN
75224,13774741,49025681,49025681,2210-10-26 13:04:00,files/p1377/p13774741/s49025681/49025681,24796442.0,39316677.0,NaN,0,1,...,SOUTH AMERICAN,2210-10-26 10:39:00,2210-10-26 23:05:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN
75225,13774741,41095220,41095220,2210-10-26 21:06:00,files/p1377/p13774741/s41095220/41095220,24796442.0,39316677.0,NaN,1,1,...,SOUTH AMERICAN,2210-10-26 10:39:00,2210-10-26 23:05:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
ecg_vitals[ecg_vitals['charttime_ed'].isna() == False]

,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,in_hosp,in_ed,...,race,ed_intime,ed_outtime,charttime_ed,ED Temperature (F),ED Heart Rate,ED Respitory Rate,ED Oxygen Saturation %,ED sbp,ED dbp
1,10101340,42195598,42195598,2110-01-18 09:41:00,files/p1010/p10101340/s42195598/42195598,22311569.0,35530181.0,NaN,0,1,...,WHITE,2110-01-18 09:28:00,2110-01-18 21:40:00,2110-01-18 09:29:00,98.2,98.0,20.0,94.0,170.0,116.0
2,10101340,48109767,48109767,2110-01-18 10:19:00,files/p1010/p10101340/s48109767/48109767,22311569.0,35530181.0,NaN,0,1,...,WHITE,2110-01-18 09:28:00,2110-01-18 21:40:00,2110-01-18 09:29:00,98.2,98.0,20.0,94.0,170.0,116.0
3,10101340,48220829,48220829,2110-01-18 14:47:00,files/p1010/p10101340/s48220829/48220829,22311569.0,35530181.0,NaN,0,1,...,WHITE,2110-01-18 09:28:00,2110-01-18 21:40:00,2110-01-18 14:20:00,NaN,89.0,16.0,96.0,147.0,90.0
19,11576763,46786216,46786216,2110-02-16 01:37:00,files/p1157/p11576763/s46786216/46786216,25798110.0,31377710.0,NaN,0,1,...,WHITE,2110-02-16 00:32:00,2110-02-16 05:40:00,2110-02-16 00:32:00,98.7,84.0,16.0,100.0,167.0,92.0
36,14609638,48429717,48429717,2110-03-13 10:29:00,files/p1460/p14609638/s48429717/48429717,26669049.0,34890898.0,NaN,0,1,...,ASIAN - CHINESE,2110-03-13 10:20:00,2110-03-13 19:26:00,2110-03-13 10:21:00,98.4,101.0,16.0,98.0,116.0,72.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75173,12648465,46900074,46900074,2209-05-02 00:11:00,files/p1264/p12648465/s46900074/46900074,22009525.0,32057880.0,NaN,0,1,...,BLACK/AFRICAN AMERICAN,2209-05-01 15:44:00,2209-05-02 01:38:00,2209-05-01 20:16:00,NaN,95.0,18.0,100.0,109.0,68.0
75176,13233424,42873676,42873676,2209-05-23 15:34:00,files/p1323/p13233424/s42873676/42873676,NaN,34904273.0,NaN,0,1,...,BLACK/AFRICAN AMERICAN,2209-05-23 15:26:00,2209-05-24 03:12:00,2209-05-23 15:28:00,98.6,102.0,19.0,97.0,96.0,45.0
75177,12648465,41799270,41799270,2209-05-26 11:04:00,files/p1264/p12648465/s41799270/41799270,20892088.0,36343811.0,NaN,0,1,...,BLACK/AFRICAN AMERICAN,2209-05-26 10:51:00,2209-05-26 19:50:00,2209-05-26 10:53:00,98.1,86.0,18.0,99.0,137.0,88.0
75188,12648465,47900027,47900027,2209-08-07 12:00:00,files/p1264/p12648465/s47900027/47900027,22702738.0,37922513.0,[32471162],0,1,...,BLACK/AFRICAN AMERICAN,2209-08-07 11:52:00,2209-08-07 18:26:00,2209-08-07 11:53:00,97.8,110.0,18.0,98.0,119.0,74.0


In [47]:
using_this

,subject_id,study_id,file_name,ecg_time,path,hadm_id,ed_stay_id,icu_stay_id,in_hosp,in_ed,...,ED Temperature (F),ED Heart Rate,ED Respitory Rate,ED Oxygen Saturation %,ED sbp,ED dbp,minutes_since_vitals,qrs_duration,pr_interval,qt_proxy
1,10101340,42195598,42195598,2110-01-18 09:41:00,files/p1010/p10101340/s42195598/42195598,22311569.0,35530181.0,NaN,0,1,...,98.2,98.0,20.0,94.0,170.0,116.0,12.0,99,209,400
2,10101340,48109767,48109767,2110-01-18 10:19:00,files/p1010/p10101340/s48109767/48109767,22311569.0,35530181.0,NaN,0,1,...,98.2,98.0,20.0,94.0,170.0,116.0,50.0,98,213,387
3,10101340,48220829,48220829,2110-01-18 14:47:00,files/p1010/p10101340/s48220829/48220829,22311569.0,35530181.0,NaN,0,1,...,NaN,89.0,16.0,96.0,147.0,90.0,27.0,102,207,394
19,11576763,46786216,46786216,2110-02-16 01:37:00,files/p1157/p11576763/s46786216/46786216,25798110.0,31377710.0,NaN,0,1,...,98.7,84.0,16.0,100.0,167.0,92.0,65.0,104,154,367
36,14609638,48429717,48429717,2110-03-13 10:29:00,files/p1460/p14609638/s48429717/48429717,26669049.0,34890898.0,NaN,0,1,...,98.4,101.0,16.0,98.0,116.0,72.0,8.0,115,202,358
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75173,12648465,46900074,46900074,2209-05-02 00:11:00,files/p1264/p12648465/s46900074/46900074,22009525.0,32057880.0,NaN,0,1,...,NaN,95.0,18.0,100.0,109.0,68.0,235.0,94,156,396
75176,13233424,42873676,42873676,2209-05-23 15:34:00,files/p1323/p13233424/s42873676/42873676,NaN,34904273.0,NaN,0,1,...,98.6,102.0,19.0,97.0,96.0,45.0,6.0,68,62,471
75177,12648465,41799270,41799270,2209-05-26 11:04:00,files/p1264/p12648465/s41799270/41799270,20892088.0,36343811.0,NaN,0,1,...,98.1,86.0,18.0,99.0,137.0,88.0,11.0,94,152,399
75188,12648465,47900027,47900027,2209-08-07 12:00:00,files/p1264/p12648465/s47900027/47900027,22702738.0,37922513.0,[32471162],0,1,...,97.8,110.0,18.0,98.0,119.0,74.0,7.0,56,152,351


In [39]:
ecg_vitals['minutes_since_vitals'] = (
    ecg_vitals['ecg_time'] - ecg_vitals['charttime_ed']
).dt.total_seconds() / 60

using_this = ecg_vitals[ecg_vitals['charttime_ed'].isna() == False]
using_this

#The goal of this baseline is to predict whether an ECG is labeled abnormal using only information that would realistically be available around the time the 
#ECG is recorded — ECG morphology, nearby ED vitals, and basic demographics.

#I structured the pipeline to be ECG-centric, with subject-level splitting to avoid patient leakage.

#I start with raw ECG timing and axis features, then add simple clinically interpretable engineered features like PR interval, QRS duration, and a QT proxy.

#These are linear combinations of existing ECG annotations — no learned transformations — so they’re transparent and easy to audit.

#explicitly track recency rather than implicitly leaking time

ecg_features = [
    'rr_interval',
    'p_onset', 'p_end',
    'qrs_onset', 'qrs_end',
    't_end',
    'p_axis', 'qrs_axis', 't_axis'
]

using_this['qrs_duration'] = using_this['qrs_end'] - using_this['qrs_onset']
using_this['pr_interval'] = using_this['qrs_onset'] - using_this['p_onset']
using_this['qt_proxy'] = using_this['t_end'] - using_this['qrs_onset']

engineered_ecg = ['qrs_duration', 'pr_interval', 'qt_proxy']

vital_features = [
    'ED Heart Rate',
    'ED sbp',
    'ED dbp',
    'ED Respitory Rate',
    'ED Oxygen Saturation %',
    'ED Temperature (F)',
    'minutes_since_vitals'
]

#Age is numeric, while race and gender are one-hot encoded.

demo_numeric = ['anchor_age']
demo_categorical = ['gender', 'race']



/tmp/ipykernel_1416/2402662052.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  using_this['qrs_duration'] = using_this['qrs_end'] - using_this['qrs_onset']
/tmp/ipykernel_1416/2402662052.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  using_this['pr_interval'] = using_this['qrs_onset'] - using_this['p_onset']
/tmp/ipykernel_1416/2402662052.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

In [40]:
y = using_this['abnormal_ecg']

X = using_this[
    ecg_features +
    engineered_ecg +
    vital_features +
    demo_numeric +
    demo_categorical
]

#I use a subject-level GroupShuffleSplit, so no patient appears in both train and test, even if they have multiple ECGs. avoids memorization across encounters
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=using_this['subject_id'])
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [41]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(
            handle_unknown='ignore',
            sparse=False
        ), demo_categorical),
        ('num', 'passthrough',
         ecg_features + engineered_ecg + vital_features + demo_numeric)
    ]
)



In [42]:
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(demo_categorical)

all_feature_names = list(cat_feature_names) + (
    ecg_features + engineered_ecg + vital_features + demo_numeric
)
#I’m using XGBoost as a strong non-linear baseline that handles mixed feature types well. 
#The depth and learning rate are conservative to reduce overfitting, especially given the relatively small abnormal class.

from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train_enc, y_train)

/home/syamala/.local/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [43]:
#The AUROC indicates very strong ranking performance — the model is highly effective at separating abnormal from normal ECGs across thresholds.
from sklearn.metrics import roc_auc_score, classification_report

y_pred_proba = model.predict_proba(X_test_enc)[:, 1]
y_pred = model.predict(X_test_enc)

print("AUROC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

AUROC: 0.9893303794527044
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1563
           1       0.91      0.78      0.84       201

    accuracy                           0.97      1764
   macro avg       0.94      0.89      0.91      1764
weighted avg       0.96      0.97      0.96      1764



In [44]:
#supports the idea that ECG morphology features are likely doing most of the discriminative work, which aligns with clinical expectations.
using_this.groupby('abnormal_ecg')[vital_features].mean()



,ED Heart Rate,ED sbp,ED dbp,ED Respitory Rate,ED Oxygen Saturation %,ED Temperature (F),minutes_since_vitals
abnormal_ecg,,,,,,,
0,83.622377,134.714571,75.280776,18.231805,97.855153,98.191859,36.116825
1,83.383745,133.130977,72.748441,18.582292,97.183585,98.172921,40.441061


In [45]:
baseline_acc = (y_test == 0).mean()
baseline_acc

0.8860544217687075

In [46]:
#ABLATION
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

def train_eval_model(X_train, X_test, y_train, y_test, desc):
    model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    )
    
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    print(f"\n=== {desc} ===")
    print("AUROC:", roc_auc_score(y_test, y_proba))
    print(classification_report(y_test, y_pred))


# Feature groups
ecg_idx = list(range(
    len(cat_feature_names),
    len(cat_feature_names) + len(ecg_features + engineered_ecg)
))

vital_idx = list(range(
    len(cat_feature_names) + len(ecg_features + engineered_ecg),
    len(cat_feature_names) + len(ecg_features + engineered_ecg) + len(vital_features)
))

demo_idx = list(range(
    0, len(cat_feature_names)
)) + list(range(
    len(cat_feature_names) + len(ecg_features + engineered_ecg + vital_features),
    X_train_enc.shape[1]
))

train_eval_model(
    X_train_enc, X_test_enc,
    y_train, y_test,
    "Full model (ECG + vitals + demographics)"
)

ecg_only_idx = ecg_idx

train_eval_model(
    X_train_enc[:, ecg_only_idx],
    X_test_enc[:, ecg_only_idx],
    y_train, y_test,
    "ECG-only"
)

vitals_only_idx = vital_idx

train_eval_model(
    X_train_enc[:, vitals_only_idx],
    X_test_enc[:, vitals_only_idx],
    y_train, y_test,
    "Vitals-only"
)

ecg_vitals_idx = ecg_idx + vital_idx

train_eval_model(
    X_train_enc[:, ecg_vitals_idx],
    X_test_enc[:, ecg_vitals_idx],
    y_train, y_test,
    "ECG + vitals (no demographics)"
)

ecg_demo_idx = ecg_idx + demo_idx

train_eval_model(
    X_train_enc[:, ecg_demo_idx],
    X_test_enc[:, ecg_demo_idx],
    y_train, y_test,
    "ECG + demographics (no vitals)"
)



=== Full model (ECG + vitals + demographics) ===
AUROC: 0.9893303794527044
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1563
           1       0.91      0.78      0.84       201

    accuracy                           0.97      1764
   macro avg       0.94      0.89      0.91      1764
weighted avg       0.96      0.97      0.96      1764


=== ECG-only ===
AUROC: 0.988942046007964
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1563
           1       0.92      0.76      0.83       201

    accuracy                           0.96      1764
   macro avg       0.94      0.88      0.91      1764
weighted avg       0.96      0.96      0.96      1764


=== Vitals-only ===
AUROC: 0.697889312236005
              precision    recall  f1-score   support

           0       0.89      0.99      0.94      1563
           1       0.10      0.00      0.01       201

    accuracy     